# Prepare skill-family, conditional-direction and tactical-intent supervision

Run this notebook before **27_skill_intent_target_relative_missile_hgt_nn_duel_skill_forecasting.ipynb**.
It augments the verified **notebook-24 v2 dataset** with family/direction and auxiliary intent labels;
run notebook 24 first if that dataset is not available. Set
`BVR_NN_TARGET_RELATIVE_MISSILE_DATA` to select an existing prepared base explicitly.
No simulator collection, feature-map duplication or new split/scaling fit is needed.

The original **33-skill targets** remain unchanged. A frozen mapping adds one of
**offensive, defensive, context_dependent** at each exact +1/+2/+5-second horizon.
Missing skill labels retain their independent `-1` masks in the intent targets.
This is a grouping of catalogue skill labels. It does not infer the aircraft's
unobserved tactical intent from a generic climb, turn, search or crank.

The **v2 label overlay** contains `intent_targets.npy`, `family_targets.npy`,
`direction_targets.npy`, frozen intent and hierarchy mappings, support tables and
copied audits. All 33 original skill labels remain available. The four explicit
mirror pairs (`notch`, `beam_missile`, `beam_target`, `crank_target`) collapse to
**29 skill families**, with **left/right direction only for those four families**.
Family and intent labels retain each missing skill's `-1`; direction is `-1` for
both missing and non-directional skills. Incomplete mirror pairs are rejected.
Re-run this notebook into a fresh small overlay if an earlier intent-only overlay
exists; notebook 24's large feature map can be reused. Its manifest pins the base
directory and manifest SHA-256. Keep that base directory available: immutable
features, anchors, skill targets and sample metadata are read from it. Hash checks
reject altered base data, taxonomy changes and inconsistent masks/labels.
The original 115 inputs, own-skill indicators, 51-frame histories, grouped splits,
training-only scaling and disjoint raw training intervals are preserved.

Outputs use `artifacts/nn_duel_skill_intent_forecasting/`. Set `BVR_NN_INTENT_DATA`
to override the intent dataset directory. Run All prepares and verifies labels;
it does not train a network. Install `.[ml]` and a Jupyter/IPython kernel.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
from bvr_behavior_prediction import skill_intent_forecasting as intent_data

EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_skill_intent_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
BASE_ROOT = ROOT / "artifacts/nn_duel_target_relative_missile_forecasting"
BASE_OVERRIDE = os.environ.get("BVR_NN_TARGET_RELATIVE_MISSILE_DATA")
if not BASE_OVERRIDE and not (BASE_ROOT / "latest_dataset.json").is_file():
    raise FileNotFoundError("Run notebook 24 first, or set BVR_NN_TARGET_RELATIVE_MISSILE_DATA.")
base_receipt = None if BASE_OVERRIDE else read_json(BASE_ROOT / "latest_dataset.json")
BASE = Path(BASE_OVERRIDE or base_receipt["directory"]).expanduser().resolve()
BASE_DIGEST = sha256(BASE / "manifest.json")
if base_receipt and BASE_DIGEST != base_receipt["manifest_sha256"]:
    raise ValueError("Base data changed since notebook 24 completed.")
PREPARATION_DIGEST = sha256(intent_data.__file__)
DATA_DEFAULT = EXPERIMENT_ROOT / "data" / f"{BASE.name}_{BASE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}_intent_hierarchy_v2"
DATA = Path(os.environ.get("BVR_NN_INTENT_DATA", str(DATA_DEFAULT))).expanduser().resolve()
print("Verified base:", BASE)
print("Intent output:", DATA)


## Freeze the label policy before training

The full 33-skill catalogue maps to **9 offensive, 11 defensive and 13 context-dependent**
entries. `lock_target` is grouped with offensive preparation; `support_missile`
with offensive weapon support. `turn_cold` and `abort_support` are defensive.
`extend`, `lag_pursuit`, target beaming/cranking, search and kinematic primitives
remain context-dependent. These are explicit modelling choices, not labels learned
from held-out outcomes. Every known catalogue entry has exactly one group; unknown
skills fail preparation instead of silently receiving a default group.

The hierarchy is also fixed before training. `skill_hierarchy.json` records original
skill order, family order, pair-branch order and left/right indices. Singleton skills
have no direction target; there is no artificial third "none" direction competing
with left/right. This partition makes reconstructed original-skill probabilities
sum to one and keeps comparisons on the original 33-class task valid.


In [ ]:
%%time
mapping_rows = [{"skill": skill, "intent": group}
                for group, skills in intent_data.INTENT_GROUPS.items() for skill in skills]
mapping = pd.DataFrame(mapping_rows).sort_values(["intent", "skill"]).reset_index(drop=True)
assert len(mapping) == mapping.skill.nunique() == 33
with pd.option_context("display.max_rows", None):
    display(mapping)
display(mapping.groupby("intent").size().rename("catalogue_skills"))

if (DATA / "manifest.json").is_file():
    manifest = read_json(DATA / "manifest.json")
    if manifest["parent_dataset"]["manifest_sha256"] != BASE_DIGEST:
        raise ValueError("Existing intent data belongs to another base; choose a fresh DATA path.")
    if manifest["intent_preparation_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Intent preparation/taxonomy code changed; choose a fresh DATA path.")
else:
    manifest = intent_data.prepare_intent_dataset(BASE, DATA)
data = intent_data.IntentNNDuelData(DATA)
assert data.features.shape[1] == 82 + len(data.classes)
assert np.array_equal(data.targets < 0, data.intent_targets < 0)
assert data.manifest["intent_supervision"] == intent_data.intent_policy(data.classes)
assert not (DATA / "features.npy").exists()
print("Feature map reused from:", data.frame_directory)
print("Intent classes:", data.intent_classes)
display(pd.DataFrame(manifest["counts"]).T)

assert data.manifest["skill_hierarchy"] == intent_data.hierarchy_policy(data.classes)
assert np.array_equal(data.targets < 0, data.family_targets < 0)
assert (data.direction_targets[data.targets < 0] == -1).all()
print("Families:", data.family_classes)
print("Paired families:", data.manifest["skill_hierarchy"]["paired_families"])
print("Direction order:", data.direction_classes)
with pd.option_context("display.max_rows", None):
    display(pd.read_csv(DATA / "skill_family_direction_mapping.csv"))


## Audit retained support and exact future labels

Support includes zero-count intent groups. No scenario is reassigned and no class
is dropped to repair support. The auxiliary labels are output supervision only;
they are never appended to input histories. Inputs continue to contain the
observing pilot's current skill, with the opposing pilot's future skills used only
as labels. The example displays one training origin and its independent horizon masks.


In [ ]:
%%time
support = pd.read_csv(DATA / "intent_support_by_split.csv")
display(support.pivot(index=["horizon_s", "intent"], columns="split", values="windows"))
sample_id = int(data.indices["train"][0])
example = data.samples.iloc[sample_id]
display(example)
display(pd.DataFrame({
    "horizon_s": HORIZONS,
    "opponent_skill": [data.classes[int(label)] if label >= 0 else "MASKED" for label in data.targets[sample_id]],
    "opponent_family": [data.family_classes[int(label)] if label >= 0 else "MASKED"
                        for label in data.family_targets[sample_id]],
    "opponent_direction": [data.direction_classes[int(label)] if label >= 0 else
                           ("MASKED" if data.targets[sample_id, h] < 0 else "not_applicable")
                           for h, label in enumerate(data.direction_targets[sample_id])],
    "opponent_intent_group": [data.intent_classes[int(label)] if label >= 0 else "MASKED"
                              for label in data.intent_targets[sample_id]],
}))
display(pd.read_csv(DATA / "family_support_by_split.csv"))
display(pd.read_csv(DATA / "direction_support_by_split.csv"))
assert data.samples.groupby("scenario_group").split.nunique().eq(1).all()
display(pd.Series(duel_data.nonoverlap_audit(data.samples)))
print("Additional intent dataset size (MiB):", sum(p.stat().st_size for p in DATA.iterdir() if p.is_file()) / 2**20)

# Publish only after the labels, parent hashes, taxonomy and masks are verified.
write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "base_directory": str(BASE), "base_manifest_sha256": BASE_DIGEST,
    "intent_policy": data.manifest["intent_supervision"]["version"],
    "skill_hierarchy": data.manifest["skill_hierarchy"]["version"],
})
print("Ready for notebook 27:", DATA)
